In [1]:

!pip install -q gradio


In [2]:

!wget -q "https://raw.githubusercontent.com/18-muskan/fashion-mnist-cnn-app/main/fashion_mnist_cnn.keras" -O fashion_mnist_cnn.keras

import os

if os.path.exists("fashion_mnist_cnn.keras"):
    print("Model file downloaded!")
    print("File size:", round(os.path.getsize("fashion_mnist_cnn.keras") / (1024 * 1024), 2), "MB")
else:
    print("Download failed. Check the GitHub model file.")


Model file downloaded!
File size: 2.61 MB


In [3]:

import gradio as gr
import numpy as np
import tensorflow as tf

from PIL import Image, ImageOps

MODEL_PATH = "fashion_mnist_cnn.keras"

class_names = [
    "T-shirt / Top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle Boot"
]

print("Loading trained CNN model...")
model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully!")

def predict_clothing(image):
    if image is None:
        return {"Please upload an image": 1.0}

    image = ImageOps.exif_transpose(image).convert("L")
    image = ImageOps.fit(image, (28, 28))

    pixels = np.array(image, dtype=np.float32)

    # Fashion MNIST uses dark backgrounds
    if pixels.mean() > 127:
        pixels = 255 - pixels

    pixels = pixels / 255.0
    pixels = pixels.reshape(1, 28, 28, 1)

    predictions = model.predict(pixels, verbose=0)[0]

    return {
        class_names[i]: float(predictions[i])
        for i in range(10)
    }

demo = gr.Interface(
    fn=predict_clothing,
    inputs=gr.Image(type="pil", label="Upload Clothing Image"),
    outputs=gr.Label(num_top_classes=3, label="CNN Predictions"),
    title="Fashion MNIST CNN Classifier",
    description=(
        "Upload a clothing image to predict its category. "
        "The model was trained on Fashion MNIST."
    )
)

demo.launch(share=True)


Loading trained CNN model...
Model loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5a355d548a7a6987c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
